In [3]:
import pandas as pd
import re

# ==================================================
# 1. LOAD DATASET
# ==================================================

INPUT_PATH = r"E:\MAJOR PROJECT\mhealth_pubmed_dataset.csv"

df = pd.read_csv(INPUT_PATH)

print("\n========== ORIGINAL DATASET ==========")
print("Number of papers:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())


# ==================================================
# 2. REMOVE DUPLICATE PAPERS
# ==================================================

before_duplicates = len(df)

df = df.drop_duplicates(subset=["pmid"])

after_duplicates = len(df)

print("\nDuplicates removed:",
      before_duplicates - after_duplicates)


# ==================================================
# 3. HANDLE MISSING TITLE AND ABSTRACT
# ==================================================

# Replace missing values with empty strings
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["abstract"] = df["abstract"].fillna("").astype(str).str.strip()

before_missing = len(df)

# Keep only papers having both title and abstract
df = df[
    (df["title"] != "") &
    (df["abstract"] != "")
]

after_missing = len(df)

print("Rows removed due to missing title/abstract:",
      before_missing - after_missing)


# ==================================================
# 4. CLEAN THE YEAR COLUMN
# ==================================================

# Example:
# "2026 Jan" -> "2026"
# "2025" -> "2025"

df["year"] = df["year"].astype(str).str.extract(r"(\d{4})")


# ==================================================
# 5. BASIC TEXT CLEANING FUNCTION
# ==================================================

def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove unnecessary special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ==================================================
# 6. CREATE ORIGINAL COMBINED DOCUMENT
# ==================================================

# Keep meaningful original text for BERTopic /
# Sentence Transformer later

df["document"] = (
    df["title"] + ". " + df["abstract"]
)


# ==================================================
# 7. CREATE CLEANED TEXT
# ==================================================

df["cleaned_text"] = df["document"].apply(clean_text)


# ==================================================
# 8. FINAL CLEANING
# ==================================================

df = df.drop_duplicates(subset=["document"])

df = df.reset_index(drop=True)


# ==================================================
# 9. SAVE CLEANED DATASET
# ==================================================

OUTPUT_PATH = r"E:\MAJOR PROJECT\mhealth_cleaned_dataset.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)


# ==================================================
# 10. DISPLAY RESULTS
# ==================================================

print("\n========== FINAL DATASET ==========")
print("Final number of papers:", len(df))

print("\nDataset saved successfully!")
print(OUTPUT_PATH)

print("\n========== SAMPLE ==========")

print(
    df[
        ["pmid", "title", "year", "document", "cleaned_text"]
    ].head(3)
)


========== ORIGINAL DATASET ==========
Number of papers: 491

Columns:
['pmid', 'title', 'abstract', 'year', 'mesh_terms']

Missing values:
pmid            0
title           0
abstract        0
year            0
mesh_terms    279
dtype: int64

Duplicates removed: 0
Rows removed due to missing title/abstract: 0

========== FINAL DATASET ==========
Final number of papers: 491

Dataset saved successfully!
E:\MAJOR PROJECT\mhealth_cleaned_dataset.csv

========== SAMPLE ==========
       pmid                                              title  year  \
0  42666381  A user centered evaluation framework for mobil...  2026   
1  42666156  Scale-up of the Diactive-1 mHealth program int...  2026   
2  42663398  Telehealth and maternal referral services: A s...  2026   

                                            document  \
0  A user centered evaluation framework for mobil...   
1  Scale-up of the Diactive-1 mHealth program int...   
2  Telehealth and maternal referral services: A s...   

    